In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

Cloning into 'capstone_project_GroupA'...
remote: Enumerating objects: 1396, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 1396 (delta 11), reused 18 (delta 9), pack-reused 1350 (from 3)
Receiving objects: 100% (1396/1396), 375.19 MiB | 23.05 MiB/s, done.
Resolving deltas: 100% (746/746), done.
Updating files: 100% (151/151), done.


In [3]:
%cd capstone_project_GroupA
!git checkout main

/content/capstone_project_GroupA
Already on 'main'
Your branch is up to date with 'origin/main'.


In [4]:
%cd src

/content/capstone_project_GroupA/src


In [5]:
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/capstone_project_GroupA/sarimax_tuning'

Run from here if not using Colab.

**Do not use specific_output_dir**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from datetime import datetime
from ModelFiles.GroupAModels import SarimaxModel
from ModelFiles.ModelConfigs import SARIMAXConfig, HORIZONS
from ModelFiles.ModelPlots import *

CONTEXT_LENGTHS = [720, 1440, 2880, 5760] # 15 days, 30 days, 60 days, 120 days.
USE_LOG_TARGET = True
DEBUG = False
EVAL_STEP_SIZE = 48
# For SARIMAX, we are not doing multiple seeds since fitting is deterministic.
for horizon in HORIZONS:
    for context_length in CONTEXT_LENGTHS:
            sarimax_config = SARIMAXConfig(
                task_id=f"sarimax_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                forecast_horizon=horizon,
                lookback_window=context_length, # For SARIMAX, this is the number of most recent time steps to use for training.
                target_col='LOG_TOTALDEMAND' if USE_LOG_TARGET else 'TOTALDEMAND',
                used_log_target=USE_LOG_TARGET,
                feature_cols=['demand_1_week_ago', 'demand_1_year_ago', 'TEMPERATURE','TEMP_SQUARED', 'IS_WEEKEND'],
                scale=True,
                # will perform grid search if any of the following parameters have more than 1 element
                p=[5,4,3],
                d=[0],
                q=[0],
                P=[1],
                D=[1],
                Q=[1],
                seasonality_period=48,
                enforce_stationarity=True,
                enforce_invertibility=True,
                seed=None,
                save_training_log= True,
                save_test_results= False,
                eval_step_size=EVAL_STEP_SIZE,
                debug=DEBUG
            )
            sarimax_model = SarimaxModel(sarimax_config, specific_output_dir=SAVE_PATH)
            sarimax_model.train_model()
            print("=" * 200)
            print("\n")